# NLSY79 Intelligence and Income Prediction


This project examines how cognitive ability, education, gender, family background, and household characteristics relate to later-life income using data from the National Longitudinal Survey of Youth (NLSY79).

The analysis combines:

- Principal Component Analysis (PCA) on ten ASVAB subtests
- Linear regression
- Decision trees
- Tree pruning
- Bootstrap aggregation
- Random forests
- Ensemble modeling

The project addresses three main questions:

1. Are ASVAB-derived measures of cognitive ability associated with later income?
2. Is there evidence of a gender income gap after controlling for measured ability and education?
3. Which predictive model performs best for forecasting future income?

> **Collaboration:** This project was completed with Keerthy Rangan and Emma Carrier for QTM 347 at Emory University.

## 1. Setup and Imports

The original course notebook included repeated imports, assignment prompts, and Google Colab upload code. This portfolio version consolidates setup and uses a repository-relative data path.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

DATA_PATH = Path("../data/IQ.Full.csv")
RANDOM_STATE = 0

## 2. Load and Prepare the Data

The dataset contains demographic information, family background variables, ten ASVAB subtest scores, education, self-esteem measures, and 2005 income.

The original project treated the final observation ("Michelle") as a holdout individual for a final income prediction. The remaining observations were divided into training, testing, and validation sets.

In [ ]:
iq_data = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {iq_data.shape}")
print(f"Total missing values: {iq_data.isna().sum().sum()}")

iq_data.head()

In [ ]:
iq = iq_data.copy()

# Remove subject identifier.
iq = iq.drop(columns="Subject")

# Encode gender as in the original project: female = 0, male = 1.
iq["Gender"] = pd.Categorical(iq["Gender"], categories=["female", "male"]).codes

# Log-transform income to reduce right skew.
iq["log_Income2005"] = np.log(iq["Income2005"])
iq = iq.drop(columns="Income2005")

# Reserve the final observation for an individual prediction.
michelle = iq.iloc[-1:].copy()
analysis_df = iq.iloc[:-1].copy()

print(f"Analysis observations: {len(analysis_df):,}")

## 3. Exploratory Data Analysis

The exploratory analysis focuses on gender, race, education, parental education, and log-transformed income. In the final report, gender was nearly balanced, race was unevenly distributed, education clustered strongly around high-school completion, and the log transformation made income substantially more symmetric.

In [ ]:
race_labels = {
    1: "Hispanic",
    2: "Black",
    3: "Not Hispanic or Black"
}

eda_df = analysis_df.copy()
eda_df["RaceLabel"] = eda_df["Race"].map(race_labels)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

sns.countplot(data=eda_df, x="Gender", ax=axes[0, 0])
axes[0, 0].set_title("Gender Distribution")
axes[0, 0].set_xlabel("Gender (0 = Female, 1 = Male)")

sns.countplot(data=eda_df, x="RaceLabel", ax=axes[0, 1])
axes[0, 1].set_title("Race Distribution")
axes[0, 1].tick_params(axis="x", rotation=20)

sns.histplot(eda_df["Educ"], bins=15, ax=axes[0, 2])
axes[0, 2].set_title("Years of Education")

sns.histplot(eda_df["MotherEd"], bins=15, ax=axes[1, 0])
axes[1, 0].set_title("Mother's Education")

sns.histplot(eda_df["FatherEd"], bins=15, ax=axes[1, 1])
axes[1, 1].set_title("Father's Education")

sns.histplot(eda_df["log_Income2005"], bins=20, ax=axes[1, 2])
axes[1, 2].set_title("Log-Transformed 2005 Income")

plt.tight_layout()
plt.show()

In [ ]:
# 70% training, 20% testing, 10% validation.
train_df, temp_df = train_test_split(
    analysis_df,
    test_size=0.30,
    random_state=RANDOM_STATE
)

test_df, validation_df = train_test_split(
    temp_df,
    test_size=1/3,
    random_state=RANDOM_STATE
)

print(f"Training observations:   {len(train_df):,}")
print(f"Testing observations:    {len(test_df):,}")
print(f"Validation observations: {len(validation_df):,}")

## 4. Principal Component Analysis of ASVAB Scores

Ten ASVAB subtests are standardized and summarized using two principal components.

In the final project:

- **ASVAB_PC1** had positive loadings across all ten tests and represented broad overall ASVAB performance.
- **ASVAB_PC2** contrasted numerical/coding/verbal strengths with technical/mechanical strengths.
- PC1 explained approximately **61% of total ASVAB variance**.

In [ ]:
ASVAB_COLS = [
    "Science", "Arith", "Word", "Parag", "Numer",
    "Coding", "Auto", "Math", "Mechanic", "Elec"
]

scaler = StandardScaler()
asvab_scaled = scaler.fit_transform(analysis_df[ASVAB_COLS])

pca = PCA(n_components=2)
asvab_components = pca.fit_transform(asvab_scaled)

analysis_df["ASVAB_PC1"] = asvab_components[:, 0]
analysis_df["ASVAB_PC2"] = asvab_components[:, 1]

loadings = pd.DataFrame(
    pca.components_.T,
    index=ASVAB_COLS,
    columns=["PC1", "PC2"]
)

print("Variance explained:")
display(
    pd.Series(
        pca.explained_variance_ratio_,
        index=["PC1", "PC2"],
        name="Explained Variance Ratio"
    ).to_frame()
)

print("ASVAB loadings:")
display(loadings)

In [ ]:
# Attach PCA scores to the previously created train/test/validation partitions.
pc_scores = analysis_df[["ASVAB_PC1", "ASVAB_PC2"]]

train_df = train_df.join(pc_scores)
test_df = test_df.join(pc_scores)
validation_df = validation_df.join(pc_scores)

# Project Michelle into the same PCA space.
michelle_scaled = scaler.transform(michelle[ASVAB_COLS])
michelle_pcs = pca.transform(michelle_scaled)

michelle["ASVAB_PC1"] = michelle_pcs[:, 0]
michelle["ASVAB_PC2"] = michelle_pcs[:, 1]

## 5. Linear Regression: Cognitive Ability and Income

An OLS model predicts log income using both ASVAB principal components while controlling for gender and education.

The final report found that both ASVAB components, gender, and education were statistically significant predictors. The model had an R² of approximately **0.217**.

In [ ]:
regression_features = ["ASVAB_PC1", "ASVAB_PC2", "Gender", "Educ"]

X_ols = sm.add_constant(analysis_df[regression_features])
y_ols = analysis_df["log_Income2005"]

ols_model = sm.OLS(y_ols, X_ols).fit()
print(ols_model.summary())

### Interpretation

The final analysis found positive associations between both ASVAB principal components and income after controlling for gender and education. Education was also positively associated with income.

Gender showed a large, statistically significant coefficient in the fitted model. Because the data are observational, this result should be interpreted as an association within the sample rather than a causal estimate of discrimination.

## 6. Decision Tree Models

### 6.1 Fit 1 — Unpruned Decision Tree

The first tree uses only gender and education. It serves as a simple baseline model.

In [ ]:
small_features = ["Gender", "Educ"]

X_train_small = train_df[small_features]
X_test_small = test_df[small_features]
X_validation_small = validation_df[small_features]

y_train = train_df["log_Income2005"]
y_test = test_df["log_Income2005"]
y_validation = validation_df["log_Income2005"]

fit1 = DecisionTreeRegressor(random_state=RANDOM_STATE)
fit1.fit(X_train_small, y_train)

fit1_test_pred = fit1.predict(X_test_small)
fit1_test_mse = mean_squared_error(y_test, fit1_test_pred)

print(f"Leaves: {fit1.get_n_leaves()}")
print(f"Depth: {fit1.get_depth()}")
print(f"Test MSE: {fit1_test_mse:.4f}")

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    fit1,
    feature_names=small_features,
    filled=True,
    rounded=True,
    fontsize=7
)
plt.title("Fit 1: Unpruned Decision Tree")
plt.show()

### 6.2 Fit 2 — Pruned Decision Tree

The second tree applies pruning constraints to reduce model complexity and improve generalization.

In [ ]:
fit2 = DecisionTreeRegressor(
    min_samples_split=20,
    ccp_alpha=0.009,
    random_state=RANDOM_STATE
)
fit2.fit(X_train_small, y_train)

fit2_train_mse = mean_squared_error(y_train, fit2.predict(X_train_small))
fit2_test_mse = mean_squared_error(y_test, fit2.predict(X_test_small))

print(f"Fit 1 training MSE: {mean_squared_error(y_train, fit1.predict(X_train_small)):.4f}")
print(f"Fit 2 training MSE: {fit2_train_mse:.4f}")
print(f"Fit 1 test MSE:     {fit1_test_mse:.4f}")
print(f"Fit 2 test MSE:     {fit2_test_mse:.4f}")

In [ ]:
plt.figure(figsize=(12, 7))
plot_tree(
    fit2,
    feature_names=small_features,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Fit 2: Pruned Decision Tree")
plt.show()

### 6.3 Fit 3 — Two-Tree Bootstrap Ensemble

Two bootstrap samples are drawn from the training data, a separate decision tree is fit to each, and their predictions are averaged.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_train = len(X_train_small)

idx1 = rng.choice(n_train, size=n_train, replace=True)
idx2 = rng.choice(n_train, size=n_train, replace=True)

tree_boot_1 = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree_boot_2 = DecisionTreeRegressor(random_state=RANDOM_STATE)

tree_boot_1.fit(X_train_small.iloc[idx1], y_train.iloc[idx1])
tree_boot_2.fit(X_train_small.iloc[idx2], y_train.iloc[idx2])

fit3_test_pred = (
    tree_boot_1.predict(X_test_small) +
    tree_boot_2.predict(X_test_small)
) / 2

fit3_test_mse = mean_squared_error(y_test, fit3_test_pred)
print(f"Fit 3 test MSE: {fit3_test_mse:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

plot_tree(
    tree_boot_1,
    feature_names=small_features,
    filled=True,
    rounded=True,
    fontsize=6,
    ax=axes[0]
)
axes[0].set_title("Bootstrap Tree 1")

plot_tree(
    tree_boot_2,
    feature_names=small_features,
    filled=True,
    rounded=True,
    fontsize=6,
    ax=axes[1]
)
axes[1].set_title("Bootstrap Tree 2")

plt.tight_layout()
plt.show()

## 7. Random Forest

### 7.1 Predictor Set

The random forest uses demographic, household, education, family-income, and ASVAB principal-component predictors. Raw ASVAB scores and self-esteem items are excluded to avoid redundancy with the PCA representation.

In [ ]:
RF_FEATURES = [
    "Imagazine",
    "Inewspaper",
    "Ilibrary",
    "MotherEd",
    "FatherEd",
    "FamilyIncome78",
    "Race",
    "Gender",
    "Educ",
    "ASVAB_PC1",
    "ASVAB_PC2",
]

X_train_rf = train_df[RF_FEATURES]
X_test_rf = test_df[RF_FEATURES]
X_validation_rf = validation_df[RF_FEATURES]

### 7.2 Tune Number of Trees

In [ ]:
tree_counts = [1, 5, 10, 25, 50, 100, 200, 250]
oob_mse_by_trees = []
test_mse_by_trees = []

for n_trees in tree_counts:
    rf = RandomForestRegressor(
        n_estimators=n_trees,
        max_features="sqrt",
        random_state=1,
        oob_score=True,
        n_jobs=-1
    )
    rf.fit(X_train_rf, y_train)

    oob_mse_by_trees.append(
        mean_squared_error(y_train, rf.oob_prediction_)
    )
    test_mse_by_trees.append(
        mean_squared_error(y_test, rf.predict(X_test_rf))
    )

plt.figure(figsize=(9, 5))
plt.plot(tree_counts, oob_mse_by_trees, marker="o", label="OOB MSE")
plt.plot(tree_counts, test_mse_by_trees, marker="o", label="Test MSE")
plt.xlabel("Number of Trees")
plt.ylabel("MSE")
plt.title("Random Forest Error vs. Number of Trees")
plt.legend()
plt.show()

### 7.3 Tune `max_features`

In [ ]:
mtry_values = list(range(1, len(RF_FEATURES) + 1))
oob_mse_by_mtry = []

for mtry in mtry_values:
    rf = RandomForestRegressor(
        n_estimators=250,
        max_features=mtry,
        random_state=1,
        oob_score=True,
        n_jobs=-1
    )
    rf.fit(X_train_rf, y_train)

    oob_mse_by_mtry.append(
        mean_squared_error(y_train, rf.oob_prediction_)
    )

best_mtry = mtry_values[int(np.argmin(oob_mse_by_mtry))]

plt.figure(figsize=(9, 5))
plt.plot(mtry_values, oob_mse_by_mtry, marker="o")
plt.axvline(best_mtry, linestyle="--", label=f"Best max_features = {best_mtry}")
plt.xlabel("max_features")
plt.ylabel("OOB MSE")
plt.title("Random Forest max_features Tuning")
plt.legend()
plt.show()

print(f"Optimal max_features: {best_mtry}")

### 7.4 Fit 4 — Tuned Random Forest

In [ ]:
fit4 = RandomForestRegressor(
    n_estimators=250,
    max_features=best_mtry,
    random_state=RANDOM_STATE,
    oob_score=True,
    n_jobs=-1
)
fit4.fit(X_train_rf, y_train)

fit4_test_pred = fit4.predict(X_test_rf)
fit4_test_mse = mean_squared_error(y_test, fit4_test_pred)
fit4_oob_mse = mean_squared_error(y_train, fit4.oob_prediction_)

print(f"OOB MSE: {fit4_oob_mse:.4f}")
print(f"Test MSE: {fit4_test_mse:.4f}")

## 8. Fit 5 — Four-Model Ensemble

The final model averages predictions from:

1. the unpruned decision tree,
2. the pruned decision tree,
3. the two-tree bootstrap ensemble, and
4. the tuned random forest.

The final report identified this ensemble as the best-performing model on the test set.

In [ ]:
fit1_test_pred = fit1.predict(X_test_small)
fit2_test_pred = fit2.predict(X_test_small)
fit3_test_pred = (
    tree_boot_1.predict(X_test_small) +
    tree_boot_2.predict(X_test_small)
) / 2
fit4_test_pred = fit4.predict(X_test_rf)

fit5_test_pred = np.column_stack([
    fit1_test_pred,
    fit2_test_pred,
    fit3_test_pred,
    fit4_test_pred
]).mean(axis=1)

fit5_test_mse = mean_squared_error(y_test, fit5_test_pred)

model_comparison = pd.DataFrame({
    "Model": [
        "Fit 1: Unpruned Tree",
        "Fit 2: Pruned Tree",
        "Fit 3: Two-Tree Bagging",
        "Fit 4: Random Forest",
        "Fit 5: Four-Model Ensemble",
    ],
    "Test MSE": [
        fit1_test_mse,
        fit2_test_mse,
        fit3_test_mse,
        fit4_test_mse,
        fit5_test_mse,
    ]
}).sort_values("Test MSE")

model_comparison

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(
    data=model_comparison,
    x="Test MSE",
    y="Model"
)
plt.title("Model Comparison")
plt.show()

## 9. Validation Performance

The ensemble is evaluated once more on the held-out validation set to assess out-of-sample performance.

In [ ]:
fit1_val = fit1.predict(X_validation_small)
fit2_val = fit2.predict(X_validation_small)
fit3_val = (
    tree_boot_1.predict(X_validation_small) +
    tree_boot_2.predict(X_validation_small)
) / 2
fit4_val = fit4.predict(X_validation_rf)

fit5_val = np.column_stack([
    fit1_val,
    fit2_val,
    fit3_val,
    fit4_val
]).mean(axis=1)

fit5_validation_mse = mean_squared_error(y_validation, fit5_val)

print(f"Fit 5 validation MSE: {fit5_validation_mse:.4f}")

## 10. Final Prediction for Michelle

The final ensemble is used to predict income for the held-out individual. The project report produced an estimated annual income of approximately **$19,250**.

In [ ]:
michelle_small = michelle[small_features]
michelle_rf = michelle[RF_FEATURES]

m1 = fit1.predict(michelle_small)[0]
m2 = fit2.predict(michelle_small)[0]
m3 = (
    tree_boot_1.predict(michelle_small)[0] +
    tree_boot_2.predict(michelle_small)[0]
) / 2
m4 = fit4.predict(michelle_rf)[0]

michelle_fit5_log = np.mean([m1, m2, m3, m4])
michelle_fit5_income = np.exp(michelle_fit5_log)

print(f"Predicted log income: {michelle_fit5_log:.4f}")
print(f"Predicted annual income: ${michelle_fit5_income:,.2f}")

## 11. Key Findings

- PCA reduced ten ASVAB subtests to two interpretable components, with PC1 capturing broad overall performance and explaining approximately 61% of ASVAB variance.
- Both ASVAB principal components were significantly associated with log income in the final regression model after controlling for gender and education.
- Education showed a positive association with later income.
- A substantial gender-income association remained after controlling for measured ability and education.
- The four-model ensemble achieved the lowest test error among the compared models in the final project.
- The final ensemble predicted approximately $19,250 in annual income for the held-out individual.

## 12. Limitations

- The dataset is observational, so estimated relationships should be interpreted as associations rather than causal effects.
- Income depends on many unobserved factors not represented in the available data.
- Occupation, industry, labor-market conditions, social networks, and other potentially important predictors are not included.
- The final ensemble's validation error was higher than its test error, illustrating uncertainty in out-of-sample prediction.